# 🟠 SafeRoute India — Notebook 02: Explore Accident Data

Explores road accident datasets and blackspot GPS points used to score routes.

**Before running:** `python src/01_download_data.py` then `python src/03_clean_accidents.py`

**Contents:**
1. Load and inspect raw accident CSVs
2. State-wise accident totals and fatality rates
3. Accident blackspot GPS points — map overlay
4. Year-on-year accident trends
5. Accident score normalisation check
6. Cross-comparison: crime vs accident risk by state

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import folium
from folium.plugins import MarkerCluster, HeatMap
from pathlib import Path

from config import TARGET_CITIES

pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', '{:.3f}'.format)
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (13, 5)

DATA_RAW  = Path('../data/raw/accidents')
DATA_PROC = Path('../data/processed')
OUT       = Path('../outputs')
OUT.mkdir(exist_ok=True)
print('Ready.')

## 1. Raw Accident Files

In [ ]:
raw_files = list(DATA_RAW.glob('*.csv'))
print(f'Found {len(raw_files)} raw accident CSV files:')
for f in raw_files:
    print(f'  {f.name:<55} {f.stat().st_size/1024:>8.1f} KB')

if raw_files:
    df_raw = pd.read_csv(raw_files[0])
    print(f'\nPrimary file shape: {df_raw.shape}')
    print(f'Columns: {list(df_raw.columns)}')
    df_raw.head()

## 2. Cleaned Accident Data — State Summary

In [ ]:
clean_path = DATA_PROC / 'accidents_clean.csv'
if not clean_path.exists():
    print('Run first: python src/03_clean_accidents.py')
else:
    df_acc = pd.read_csv(clean_path)
    print(f'Shape: {df_acc.shape}')
    print(f'Columns: {list(df_acc.columns)}')
    df_acc.head(10)

In [ ]:
# Top states by accident score
if 'df_acc' in dir():
    score_col = next((c for c in df_acc.columns if 'SCORE' in c.upper()), None)
    state_col = next((c for c in df_acc.columns if 'STATE' in c.upper()), None)
    
    if score_col and state_col:
        top = df_acc.groupby(state_col)[score_col].mean().nlargest(20).reset_index()
        
        fig, ax = plt.subplots(figsize=(12, 6))
        colors = plt.cm.YlOrRd(np.linspace(0.3, 0.9, len(top)))
        ax.barh(top[state_col], top[score_col], color=colors, edgecolor='white', lw=0.3)
        ax.set_xlabel('Mean Accident Risk Score (0–1)', fontsize=11)
        ax.set_title('Top 20 States by Road Accident Risk Score', fontsize=13, fontweight='bold')
        ax.axvline(top[score_col].mean(), color='black', ls='--', lw=1.5,
                   label=f'Average = {top[score_col].mean():.3f}')
        ax.legend()
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        plt.tight_layout()
        plt.savefig(str(OUT / 'top20_accident_states.png'), dpi=150, bbox_inches='tight')
        plt.show()

## 3. Accident Blackspot GPS Points Map

In [ ]:
blackspot_path = DATA_PROC / 'accidents_clean.csv'
if blackspot_path.exists():
    df_bs = pd.read_csv(blackspot_path)
    lat_col = next((c for c in df_bs.columns if 'LAT' in c.upper()), None)
    lon_col = next((c for c in df_bs.columns if 'LON' in c.upper()), None)
    
    if lat_col and lon_col:
        df_bs = df_bs.dropna(subset=[lat_col, lon_col])
        # Filter to India
        df_bs = df_bs[
            df_bs[lat_col].between(6, 37.5) &
            df_bs[lon_col].between(67, 98)
        ]
        print(f'Blackspot points with GPS: {len(df_bs):,}')
        
        m = folium.Map(location=[20.5, 78.9], zoom_start=5, tiles='CartoDB dark_matter')
        cluster = MarkerCluster(name='Accident Blackspots')
        
        for _, row in df_bs.head(2000).iterrows():
            folium.CircleMarker(
                location=[row[lat_col], row[lon_col]],
                radius=4, color='#f59e0b', fill=True, fill_opacity=0.7,
                popup=f"Accident score: {row.get(score_col, 'N/A'):.3f}" if score_col in df_bs.columns else ''
            ).add_to(cluster)
        
        cluster.add_to(m)
        
        # City markers
        for city_key, cfg in TARGET_CITIES.items():
            folium.Marker(
                location=list(cfg['center']),
                popup=city_key.title(),
                icon=folium.Icon(color='white', icon='home')
            ).add_to(m)
        
        folium.LayerControl().add_to(m)
        out_path = str(OUT / 'accident_blackspots_india.html')
        m.save(out_path)
        print(f'Saved: {out_path}')
        m
    else:
        print('No LAT/LON columns found in accident data — GPS blackspot data may not have been downloaded.')
        print('Source: https://data.gov.in/catalog/accident-black-spots')

## 4. Accident Heatmap — All India

In [ ]:
if 'df_bs' in dir() and lat_col and lon_col and len(df_bs) > 0:
    m2 = folium.Map(location=[20.5, 78.9], zoom_start=5, tiles='CartoDB dark_matter')
    
    acc_col = next((c for c in df_bs.columns if 'ACCIDENT_SCORE' in c.upper()), None)
    if acc_col:
        heat_data = df_bs[[lat_col, lon_col, acc_col]].dropna().values.tolist()
    else:
        heat_data = df_bs[[lat_col, lon_col]].assign(w=0.6).values.tolist()
    
    HeatMap(
        heat_data, radius=16, blur=12, max_zoom=16,
        gradient={0.2:'#3b82f6', 0.5:'#f59e0b', 0.8:'#f97316', 1.0:'#dc2626'}
    ).add_to(m2)
    
    out_path = str(OUT / 'accident_heatmap_india.html')
    m2.save(out_path)
    print(f'Accident heatmap saved → {out_path}')
    m2

## 5. Accident Score Distribution

In [ ]:
if 'df_acc' in dir() and score_col:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    vals = df_acc[score_col].dropna()
    
    axes[0].hist(vals, bins=40, color='#f59e0b', alpha=0.85, edgecolor='white', lw=0.3)
    axes[0].axvline(vals.mean(),   color='black', ls='--', lw=2, label=f'Mean={vals.mean():.3f}')
    axes[0].axvline(vals.median(), color='blue',  ls=':',  lw=2, label=f'Median={vals.median():.3f}')
    axes[0].set_xlabel('Accident Risk Score (0–1)')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Accident Score Distribution')
    axes[0].legend()
    
    sorted_v = np.sort(vals)
    cdf = np.arange(1, len(sorted_v)+1) / len(sorted_v)
    axes[1].plot(sorted_v, cdf, color='#f59e0b', lw=2)
    axes[1].fill_between(sorted_v, cdf, alpha=0.15, color='#f59e0b')
    axes[1].set_xlabel('Accident Risk Score')
    axes[1].set_ylabel('Cumulative Proportion')
    axes[1].set_title('Cumulative Distribution (CDF)')
    
    for ax in axes:
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    
    plt.suptitle('Accident Score Distribution Analysis', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(str(OUT / 'accident_score_distribution.png'), dpi=150, bbox_inches='tight')
    plt.show()

## 6. Cross-Comparison: Crime vs Accident Risk by State

In [ ]:
crime_path = DATA_PROC / 'crime_clean.csv'
if crime_path.exists() and 'df_acc' in dir():
    df_crime = pd.read_csv(crime_path)
    crime_score_col = next((c for c in df_crime.columns if 'NORM' in c.upper()), None)
    
    if crime_score_col and state_col and score_col:
        crime_by_state = df_crime.groupby('STATE')[crime_score_col].mean().reset_index()
        crime_by_state.columns = ['STATE', 'CRIME_SCORE']
        
        acc_by_state = df_acc.groupby(state_col)[score_col].mean().reset_index()
        acc_by_state.columns = ['STATE', 'ACCIDENT_SCORE']
        
        merged = crime_by_state.merge(acc_by_state, on='STATE', how='inner')
        
        fig, ax = plt.subplots(figsize=(10, 8))
        scatter = ax.scatter(
            merged['CRIME_SCORE'], merged['ACCIDENT_SCORE'],
            c=merged['CRIME_SCORE'] + merged['ACCIDENT_SCORE'],
            cmap='YlOrRd', s=80, alpha=0.8, edgecolors='white', lw=0.5
        )
        
        for _, row in merged.iterrows():
            ax.annotate(row['STATE'], (row['CRIME_SCORE'], row['ACCIDENT_SCORE']),
                        fontsize=7, ha='center', va='bottom', alpha=0.8)
        
        # Diagonal reference line
        lim = max(merged['CRIME_SCORE'].max(), merged['ACCIDENT_SCORE'].max()) + 0.05
        ax.plot([0, lim], [0, lim], 'gray', ls='--', lw=1, alpha=0.5, label='Equal risk')
        
        ax.set_xlabel('Crime Risk Score (mean across districts)', fontsize=11)
        ax.set_ylabel('Accident Risk Score (mean)', fontsize=11)
        ax.set_title('Crime vs Accident Risk by State', fontsize=13, fontweight='bold')
        plt.colorbar(scatter, ax=ax, label='Combined Risk')
        ax.legend()
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        plt.tight_layout()
        plt.savefig(str(OUT / 'crime_vs_accident_scatter.png'), dpi=150, bbox_inches='tight')
        plt.show()
        
        corr = merged['CRIME_SCORE'].corr(merged['ACCIDENT_SCORE'])
        print(f'Pearson correlation (crime vs accident): {corr:.3f}')
        if abs(corr) > 0.5:
            print('→ Strong correlation: states with high crime also tend to have high accidents.')
        else:
            print('→ Weak correlation: crime and accident risks are somewhat independent.')